Clean Minimal Notebook Template for Mini-Project 3A
Author: ChatGPT
Purpose: Create 3-node FABRIC cluster + verify connectivity

This notebook removes unnecessary complexity from the original template.
It focuses only on what you need to complete the assignment.

In [ ]:
STEP 0: Install Dependencies (Run Once)

In [ ]:
# Uncomment if needed
# !pip install fabrictestbed-extensions pandas

STEP 1: Authentication + FABRIC Setup

In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

PROJECT_ID = "7db31408-b695-427c-8856-8761292228b6"

try:
    fablib = FablibManager(project_id=PROJECT_ID)
    fablib.verify_and_configure()
    fablib.save_config()
    fablib.show_config()
    print("✅ FABRIC configuration ready")
except Exception as e:
    print("Auth/setup error:", e)

User: cslgbt@missouri.edu bastion key is valid!
Configuration is valid
User: cslgbt@missouri.edu bastion key is valid!
Configuration is valid
Please save the config!


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,7db31408-b695-427c-8856-8761292228b6
Bastion Host,bastion.fabric-testbed.net
Bastion Username,cslgbt_0000451422
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


✅ FABRIC configuration ready


STEP 1A: Check for Resource Availability

Key: Look for site with:

✅ High cores_available
✅ High ram_available
✅ (Ignore GPU columns unless required)

In [3]:
try:
    resources_json = fablib.list_sites(output='json', quiet=True)

    import pandas as pd

    sites_df = pd.read_json(resources_json)

    display(
        sites_df[
            [
                "name",
                "hosts",
                "cpus",
                "cores_available",
                "ram_available",
                "tesla_t4_available",
                "rtx6000_available",
                "a30_available",
                "a40_available"
            ]
        ]
        .sort_values("cores_available", ascending=False)
    )

    print("✅ Resource check complete")

except Exception as e:
    print("Resource check error:", e)

/tmp/ipykernel_659/494044139.py:6: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sites_df = pd.read_json(resources_json)


,name,hosts,cpus,cores_available,ram_available,tesla_t4_available,rtx6000_available,a30_available,a40_available
2,EDUKY,18,36,73566,7908,0,0,0,0
13,CERN,6,12,328,1108,0,0,1,0
10,HAWI,5,10,324,1502,0,0,0,0
6,PSC,3,6,308,1130,0,0,0,0
30,TOKY,3,6,300,1250,0,0,1,0
21,GATECH,5,10,298,1186,0,0,1,0
8,CLEM,3,6,242,1074,0,0,0,0
0,FIU,5,10,237,1130,0,1,0,0
26,SALT,3,6,210,954,2,2,0,0
24,WASH,3,6,204,626,0,2,0,0


✅ Resource check complete


STEP 2: Cluster Parameters (Modify if needed)

In [7]:
NUM_NODES = 3
SITES = "EDUKY"
SLICENAME = "MiniProject3A_Cluster"

INSTANCE_WORKER = "fabric.c8.m16.d100"
INSTANCE_MASTER = "fabric.c8.m16.d100"
IMAGE = "default_ubuntu_20"

STEP 3: Create Slice Cluster

In [8]:
try:
    slice_obj = fablib.new_slice(name=SLICENAME)

    node_names = [f"Node{i+1}" for i in range(NUM_NODES)]

    iface_list = []

    for idx in range(NUM_NODES):
        if idx == 0:
            node = slice_obj.add_node(
                name=node_names[idx],
                site=SITES,
                instance_type=INSTANCE_MASTER,
                image=IMAGE
            )
        else:
            node = slice_obj.add_node(
                name=node_names[idx],
                site=SITES,
                instance_type=INSTANCE_WORKER,
                image=IMAGE
            )

        iface = node.add_component(model='NIC_Basic', name=f"nic{idx+1}")
        iface_list.append(iface.get_interfaces()[0])

    print("✅ Cluster definition complete")

except Exception as e:
    print("Cluster definition error:", e)

✅ Cluster definition complete


STEP 4: Submit Slice (Create Cluster)

In [9]:
try:
    slice_obj.submit()
    print("✅ Slice submitted. Wait for provisioning...")
except Exception as e:
    print("Slice submission error:", e)



Retry: 9, Time: 222 sec


ID,6714af76-f7e2-443c-a5f8-db522b3426ad
Name,MiniProject3A_Cluster
Lease Expiration (UTC),2026-03-04 01:45:20 +0000
Lease Start (UTC),2026-03-03 01:45:20 +0000
Project ID,7db31408-b695-427c-8856-8761292228b6
State,StableOK
Email,cslgbt@missouri.edu
UserId,7fa5070e-304e-4e07-9b7e-824b45c39b7e


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
e6798652-bee4-4d61-b3cf-e5be4846c129,Node1,4,16,10,default_ubuntu_20,qcow2,eduky-w9.fabric-testbed.net,EDUKY,ubuntu,2610:1e0:1700:206:f816:3eff:fe05:2d68,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:1e0:1700:206:f816:3eff:fe05:2d68,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
a685fe3f-c604-4f38-a2f9-6b7199795547,Node2,4,16,10,default_ubuntu_20,qcow2,eduky-w13.fabric-testbed.net,EDUKY,ubuntu,2610:1e0:1700:206:f816:3eff:fe15:62fa,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:1e0:1700:206:f816:3eff:fe15:62fa,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
3637037a-5ff4-4143-8052-cbbac9b77c23,Node3,4,16,10,default_ubuntu_20,qcow2,eduky-w1.fabric-testbed.net,EDUKY,ubuntu,2610:1e0:1700:206:f816:3eff:fef9:3384,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:1e0:1700:206:f816:3eff:fef9:3384,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


None

Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,None,100,config,,1A:46:D8:75:1E:E9,enp7s0,enp7s0,None,1,None
Node2-nic2-p1,p1,Node2,None,100,config,,06:A8:BC:8D:42:82,enp7s0,enp7s0,None,1,None
Node3-nic3-p1,p1,Node3,None,100,config,,02:57:4F:F8:DD:D4,enp7s0,enp7s0,None,1,None



Time to print interfaces 226 seconds
✅ Slice submitted. Wait for provisioning...


STEP 5: Assign IPs + Enable Network Interface

In [ ]:
from ipaddress import IPv4Network
import time

try:
    subnet = IPv4Network("192.168.1.0/24")
    available_ips = list(subnet)[1:]

    time.sleep(30)  # Give time for nodes to boot

    for node in slice_obj.get_nodes():
        iface = node.get_interface(network_name="cluster_network")
        if iface:
            ip_addr = available_ips.pop(0)
            iface.ip_addr_add(addr=ip_addr, subnet=subnet)
            node.execute(f"sudo ifconfig {iface.get_os_interface()} up")

    print("✅ Network interfaces configured")

except Exception as e:
    print("Network config error:", e)

STEP 6: Connectivity Test (Very Important Check)

In [12]:
try:
    node1 = slice_obj.get_node(name="Node1")
    stdout, stderr = node1.execute("ping -c 3 192.168.1.2")

    print("Ping Output:")
    print(stdout)
    print(stderr)

    print("✅ Cluster connectivity check complete")

except Exception as e:
    print("Connectivity test error:", e)

ping: connect: Network is unreachable
Ping Output:

ping: connect: Network is unreachable

✅ Cluster connectivity check complete


In [13]:
print(slice_obj.get_state())

StableOK


In [14]:
node1.execute("wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda.sh")
node1.execute("bash ~/miniconda.sh -b")

--2026-03-03 02:15:55--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 2606:4700::6810:bf9e, 2606:4700::6810:20f1, 104.16.191.158, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|2606:4700::6810:bf9e|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156772981 (150M) [application/octet-stream]
Saving to: ‘/home/ubuntu/miniconda.sh’

     0K .......... .......... .......... .......... ..........  0% 43.7M 3s
    50K .......... .......... .......... .......... ..........  0% 4.81M 17s
   100K .......... .......... .......... .......... ..........  0% 5.12M 21s
   150K .......... .......... .......... .......... ..........  0% 23.9M 17s
   200K .......... .......... .......... .......... ..........  0% 17.1M 16s
   250K .......... .......... .......... .......... ..........  0% 13.7M 15s
   300K .......... .......... .......... .......... ..........  0% 28.3M 14s
   350K .......... ..

('PREFIX=/home/ubuntu/miniconda3\nUnpacking bootstrapper...\nUnpacking payload...\n\nInstalling base environment...\n\nPreparing transaction: ...working... done\nExecuting transaction: ...working... done\ninstallation finished.\n',
 '')

In [15]:
node1.execute("export PATH=~/miniconda3/bin:$PATH")

('', '')

In [17]:
node1.execute("~/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main")
node1.execute("~/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r")

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


('accepted Terms of Service for https://repo.anaconda.com/pkgs/r\n', '')

In [18]:
node1.execute("~/miniconda3/bin/conda create -y -n fear_sim python=3.10")

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/ubuntu/miniconda3/envs/fear_sim

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    expat-2.7.4                |       h7354ed3_0          25 KB
    libexpat-2.7.4             |       h7354ed3_0         122 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    openssl-3.5.5              |       h1b28b03_0         5.5 MB
    packaging-25.0             |  py310h06a4308_1         164 KB
    pip-26.0.1                 |     pyhc872135_0         1.1 MB
    python-3.10.19             |       h6fa692b_0        24.5 MB
    setuptools-80.10.2         |  py310h06a4308_0         1.3 MB
    sqlite-3.51.1              |       he0a8d7e_0         1.2 MB
    tzdata-202

('2 channel Terms of Service accepted\nRetrieving notices: - \x08\x08\\ \x08\x08| \x08\x08/ \x08\x08- \x08\x08\\ \x08\x08| \x08\x08done\nChannels:\n - defaults\nPlatform: linux-64\nCollecting package metadata (repodata.json): - \x08\x08\\ \x08\x08| \x08\x08/ \x08\x08- \x08\x08\\ \x08\x08| \x08\x08/ \x08\x08done\nSolving environment: \\ \x08\x08done\n\n## Package Plan ##\n\n  environment location: /home/ubuntu/miniconda3/envs/fear_sim\n\n  added / updated specs:\n    - python=3.10\n\n\nThe following packages will be downloaded:\n\n    package                    |            build\n    ---------------------------|-----------------\n    expat-2.7.4                |       h7354ed3_0          25 KB\n    libexpat-2.7.4             |       h7354ed3_0         122 KB\n    libnsl-2.0.0               |       h5eee18b_0          31 KB\n    openssl-3.5.5              |       h1b28b03_0         5.5 MB\n    packaging-25.0             |  py310h06a4308_1         164 KB\n    pip-26.0.1                 |

In [19]:
node1.execute("source ~/miniconda3/bin/activate fear_sim && pip install neuron bmtk mpich")

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (Caused by NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f02d7db9900>: Failed to establish a new connection: [Errno 101] Network is unreachable'))



('Collecting neuron\n',
 "  WARNING: Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f02d7db8fa0>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f02d7db9240>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=2, connect=None, read=None, redirect=None

In [20]:
node1.execute("source ~/miniconda3/bin/activate fear_sim && pip install neuron bmtk mpich --no-cache-dir")

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (Caused by NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f3bc0f50fd0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))



('Collecting neuron\n',
 "  WARNING: Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f3bc0f50670>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f3bc0f50910>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=2, connect=None, read=None, redirect=None

In [21]:
node1.execute("source ~/miniconda3/bin/activate fear_sim && python -m pip install neuron")

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (Caused by NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f2a73a3a110>: Failed to establish a new connection: [Errno 101] Network is unreachable'))



('Collecting neuron\n',
 "  WARNING: Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f2a73a397b0>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f2a73a39a50>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=2, connect=None, read=None, redirect=None

In [25]:
node1.execute("source ~/miniconda3/bin/activate fear_sim && conda list")

# packages in environment at /home/ubuntu/miniconda3/envs/fear_sim:
#
# Name                     Version          Build            Channel
_libgcc_mutex              0.1              main
_openmp_mutex              5.1              1_gnu
bzip2                      1.0.8            h5eee18b_6
ca-certificates            2025.12.2        h06a4308_0
expat                      2.7.4            h7354ed3_0
ld_impl_linux-64           2.44             h153f514_2
libexpat                   2.7.4            h7354ed3_0
libffi                     3.4.4            h6a678d5_1
libgcc                     15.2.0           h69a1729_7
libgcc-ng                  15.2.0           h166f726_7
libgomp                    15.2.0           h4751f2c_7
libnsl                     2.0.0            h5eee18b_0
libstdcxx                  15.2.0           h39759b7_7
libstdcxx-ng               15.2.0           hc03a8fd_7
libuuid                    1.41.5           h5eee18b_0
libxcb                     1.17.0           h9b

('# packages in environment at /home/ubuntu/miniconda3/envs/fear_sim:\n#\n# Name                     Version          Build            Channel\n_libgcc_mutex              0.1              main\n_openmp_mutex              5.1              1_gnu\nbzip2                      1.0.8            h5eee18b_6\nca-certificates            2025.12.2        h06a4308_0\nexpat                      2.7.4            h7354ed3_0\nld_impl_linux-64           2.44             h153f514_2\nlibexpat                   2.7.4            h7354ed3_0\nlibffi                     3.4.4            h6a678d5_1\nlibgcc                     15.2.0           h69a1729_7\nlibgcc-ng                  15.2.0           h166f726_7\nlibgomp                    15.2.0           h4751f2c_7\nlibnsl                     2.0.0            h5eee18b_0\nlibstdcxx                  15.2.0           h39759b7_7\nlibstdcxx-ng               15.2.0           hc03a8fd_7\nlibuuid                    1.41.5           h5eee18b_0\nlibxcb                     

In [27]:
node1.execute(
"source ~/miniconda3/bin/activate fear_sim && python -m pip install neuron bmtk mpich --no-cache-dir"
)

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (Caused by NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f5a35825210>: Failed to establish a new connection: [Errno 101] Network is unreachable'))



('Collecting neuron\n',
 "  WARNING: Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f5a358248b0>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f5a35824b50>: Failed to establish a new connection: [Errno 101] Network is unreachable')': /packages/12/da/36e176d6598397f4c168d950cb46a75721961d526abde8aae1c818bb4462/neuron-9.0.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata\n  WARNING: Retrying (Retry(total=2, connect=None, read=None, redirect=None

In [30]:
node1.execute("source ~/miniconda3/bin/activate fear_sim && python -c 'import sys; print(\"Python working\")'")

Python working


('Python working\n', '')

In [31]:
node1.execute("git clone https://github.com/cyneuro/CI-BioEng-Class.git")

Cloning into 'CI-BioEng-Class'...


('', "Cloning into 'CI-BioEng-Class'...\n")

In [32]:
node1.execute("cd CI-BioEng-Class && ls")

README.md
cs4001_allen.ipynb
cs4001_git.ipynb
cs4001_mpi.ipynb
cs4001_terminal.ipynb
emotion_recognition
fear_simulation


('README.md\ncs4001_allen.ipynb\ncs4001_git.ipynb\ncs4001_mpi.ipynb\ncs4001_terminal.ipynb\nemotion_recognition\nfear_simulation\n',
 '')